In [208]:
import gdown

gdown.download('https://bit.ly/3RhoNho', 'ns_202104.csv', quiet=False)

import pandas as pd
ns_df = pd.read_csv('ns_202104.csv', low_memory=False)

Downloading...
From: https://bit.ly/3RhoNho
To: c:\data\ns_202104.csv
100%|██████████| 57.6M/57.6M [00:05<00:00, 10.1MB/s]


In [ ]:
# 열삭제
ns_df.head()
ns_df2 = ns_df.dropna(how='all', axis=1)
ns_df2

,번호,도서명,저자,출판사,발행년도,ISBN,세트 ISBN,부가기호,권,도서권수,대출건수,등록일자
0,1,인공지능과 흙,김동훈 지음,민음사,2021,9788937444319,NaN,NaN,NaN,1,0,2021-03-19
1,2,가짜 행복 권하는 사회,김태형 지음,갈매나무,2021,9791190123969,NaN,NaN,NaN,1,0,2021-03-19
2,3,나도 한 문장 잘 쓰면 바랄 게 없겠네,김선영 지음,블랙피쉬,2021,9788968332982,NaN,NaN,NaN,1,0,2021-03-19
3,4,예루살렘 해변,"이도 게펜 지음, 임재희 옮김",문학세계사,2021,9788970759906,NaN,NaN,NaN,1,0,2021-03-19
4,5,김성곤의 중국한시기행 : 장강·황하 편,김성곤 지음,김영사,2021,9788934990833,NaN,NaN,NaN,1,0,2021-03-19
...,...,...,...,...,...,...,...,...,...,...,...,...
401677,401678,韓國現代詩大系,채만묵 編著,한국문화사,1996,9788977352971,9788977352988,NaN,3,1,0,1970-01-01
401678,401679,뉴 웨이브,제임스 모나코 지음,한나래,1996,9788985367448,9788985367424,NaN,2,1,0,1970-01-01
401679,401680,(최인훈 장편소설)화두,최인훈 지음,민음사,1994,9788937401596,9788937401589,NaN,2,1,0,1970-01-01
401680,401681,독일 문학과 세계 문학,吳漢鎭 編著,벽호,1995,9788947700368,9788947700405,NaN,3,2,0,1970-01-01


In [221]:
#행삭제
ns_df
ns_df.drop(['주제분류번호', 'unnamed: 13'], axis=1, inplace=True)


KeyError: "['주제분류번호', 'unnamed: 13'] not found in axis"

In [ ]:
#중복된 행 찾기 .duplicated
dup_rows = ns_df.duplicated(subset=['도서명', '저자', 'ISBN'])
ns_df_dup = ns_df[dup_rows]

#중복된 행 그룹으로 묶어서 대출건수 계산하기
ns_group = ns_df[['도서명', '저자', 'ISBN', '권', '대출건수']]
ns_group_df = ns_group.groupby(by = ['도서명', '저자', 'ISBN', '권'], dropna=False).sum()

#(원본) 데이터프레임 중복행 수정
dup_rows2 = ns_df.duplicated(['도서명', '저자', '권', 'ISBN'])
dup_rows3 = ~dup_rows
ns_book = ns_df[dup_rows3].copy()

#(원본)데이터프레임 중복행 수정 쉬운 버전
ns_book_easy = ns_df.drop_duplicates(['도서명', '저자', '권', 'ISBN']).copy()
sum(ns_book_easy.duplicated(['도서명', '저자', '권', 'ISBN']))

#원본 데이터프레임 인덱스를 수정버전 인덱스로 설정하기
ns_book.set_index(['도서명', '저자', 'ISBN', '권'], inplace=True)
ns_book.update(ns_group_df)
ns_book2 = ns_book.reset_index()

,도서명,저자,ISBN,권,번호,출판사,발행년도,세트 ISBN,부가기호,도서권수,대출건수,등록일자
0,인공지능과 흙,김동훈 지음,9788937444319,NaN,1,민음사,2021,NaN,NaN,1,0,2021-03-19
1,가짜 행복 권하는 사회,김태형 지음,9791190123969,NaN,2,갈매나무,2021,NaN,NaN,1,0,2021-03-19
2,나도 한 문장 잘 쓰면 바랄 게 없겠네,김선영 지음,9788968332982,NaN,3,블랙피쉬,2021,NaN,NaN,1,0,2021-03-19
3,예루살렘 해변,"이도 게펜 지음, 임재희 옮김",9788970759906,NaN,4,문학세계사,2021,NaN,NaN,1,0,2021-03-19
4,김성곤의 중국한시기행 : 장강·황하 편,김성곤 지음,9788934990833,NaN,5,김영사,2021,NaN,NaN,1,0,2021-03-19
...,...,...,...,...,...,...,...,...,...,...,...,...
379581,소설의 사회사 비교론,조동일 지음,9788942340262,3,401674,지식산업사,2001,9788942300365,NaN,1,0,1970-01-01
379582,큰오빠,박정근 지음,9788974231323,2,401677,우리문학사,1998,9788974231309,NaN,2,0,1970-01-01
379583,韓國現代詩大系,채만묵 編著,9788977352971,3,401678,한국문화사,1996,9788977352988,NaN,1,0,1970-01-01
379584,뉴 웨이브,제임스 모나코 지음,9788985367448,2,401679,한나래,1996,9788985367424,NaN,1,0,1970-01-01


In [289]:
ns_book2 = ns_book2[ns_df.columns]
ns_book2.to_csv(r"c:/DATA/ns_book2.csv", index=False, encoding='utf-8')